# QUEST Decision Tree Implementation

## Quick, Unbiased, Efficient Statistical Tree

This notebook implements the QUEST algorithm as described in the original paper:

> Loh, W.-Y., & Shih, Y.-S. (1997). *Split selection methods for classification trees*. Statistica Sinica, 7, 815-840.

### Why QUEST?

QUEST was developed to address a fundamental limitation of earlier tree algorithms like CART and C4.5:

- **CART** has selection bias toward variables with many possible splits
- **C4.5** has bias toward variables with many categories

QUEST achieves **unbiased variable selection** by separating the variable selection step from the split point selection step, using statistical hypothesis tests.

### Key Features

1. **Unbiased Variable Selection**: Uses ANOVA F-tests and chi-square tests
2. **Efficient Split Finding**: Uses quadratic discriminant analysis (QDA)
3. **Handles Mixed Data**: Transforms categorical variables via CRIMCOORDS
4. **Multiclass Support**: Uses Super-Class Clustering (Algorithm 4) to reduce K-class problems to binary
5. **Cost-Complexity Pruning**: CART-style post-pruning for optimal tree size

### Algorithm Components (from the paper)

- **Algorithm 1**: Split point selection using QDA
- **Algorithm 2**: CRIMCOORDS transformation for categorical variables
- **Algorithm 3**: Variable selection via statistical tests (ANOVA/Chi-square)
- **Algorithm 4**: Super-class clustering for multiclass problems

---
## 1. Imports & Constants


In [1]:
import numpy as np
from scipy import stats
from scipy import linalg
from scipy.cluster.vq import kmeans2
import warnings

# =============================================================================
# CONSTANTS
# =============================================================================

# Significance level for hypothesis tests (Bonferroni-corrected internally)
ALPHA = 0.05

# Minimum number of samples required to attempt a split
MIN_SAMPLES_SPLIT = 10

# Maximum depth of the tree (used as safety limit, not primary stopping criterion)
# QUEST relies on cost-complexity pruning, so this is set high
MAX_DEPTH = 20

# Minimum samples per leaf (for tree growing phase)
MIN_SAMPLES_LEAF = 5

# Cost-complexity pruning parameter (alpha)
# Set to 0 for no pruning; optimal value typically found via cross-validation
DEFAULT_CCP_ALPHA = 0.0

# Random seed for reproducibility
np.random.seed(42)

print("QUEST Decision Tree - Configuration")
print("=" * 50)
print(f"  ALPHA (significance level): {ALPHA}")
print(f"  MIN_SAMPLES_SPLIT: {MIN_SAMPLES_SPLIT}")
print(f"  MIN_SAMPLES_LEAF: {MIN_SAMPLES_LEAF}")
print(f"  MAX_DEPTH: {MAX_DEPTH} (safety limit)")
print(f"  DEFAULT_CCP_ALPHA: {DEFAULT_CCP_ALPHA}")
print("=" * 50)
print("\nNote: QUEST uses cost-complexity pruning as the primary")
print("regularization method, not max_depth or min_samples.")

QUEST Decision Tree - Configuration
  ALPHA (significance level): 0.05
  MIN_SAMPLES_SPLIT: 10
  MIN_SAMPLES_LEAF: 5
  MAX_DEPTH: 20 (safety limit)
  DEFAULT_CCP_ALPHA: 0.0

Note: QUEST uses cost-complexity pruning as the primary
regularization method, not max_depth or min_samples.


---
## 2. Helper Math Functions

These functions implement the core statistical procedures described in the QUEST paper.

---
### Block A — Variable Selection (Algorithm 3)

The variable selection procedure in QUEST uses statistical tests to select the "best" variable at each node:

1. For **continuous** variables: ANOVA F-test
2. For **categorical** variables: Chi-square test of independence
3. If no variable is significant: Levene's test for equality of variances

The key insight is that these tests have **approximately equal power** across different variable types, eliminating selection bias.

---
### Super-Class Clustering (Algorithm 4) — Multiclass Handling

QUEST handles K > 2 classes by converting the problem to binary at each node:

1. Compute the **class centroids** (mean vector for each class)
2. Apply **2-means clustering** on the K centroids (not on samples)
3. This partitions the K classes into two **super-classes** (Group A, Group B)
4. Temporarily relabel samples: Group A → 0, Group B → 1
5. Apply standard binary QDA/CRIMCOORDS on the relabeled data

This approach ensures:
- The binary splitting machinery (QDA, CRIMCOORDS) works unchanged
- All K classes contribute to the split decision
- No class is arbitrarily ignored

In [2]:
def compute_superclass_labels(X, y, feature_types=None):
    """
    Algorithm 4: Super-Class Clustering for Multiclass Problems.

    QUEST converts K-class problems to binary by clustering class centroids
    into two super-classes. This preserves the binary QDA/CRIMCOORDS machinery
    while properly handling all K classes.

    Mathematical Formulation (from QUEST paper):
    -------------------------------------------
    1. For each class k ∈ {1, ..., K}, compute the centroid:
       μ_k = (1/n_k) Σ_{i: y_i = k} x_i

       For categorical features, we use the mode or proportion vector.

    2. Apply 2-means clustering to the K centroids {μ_1, ..., μ_K}:
       - Initialize two cluster centers
       - Iterate until convergence
       - Each class is assigned to one of two groups

    3. Output binary labels:
       - Samples from classes in Group A → 0
       - Samples from classes in Group B → 1

    Parameters
    ----------
    X : np.ndarray
        Feature matrix (n_samples, n_features)
    y : np.ndarray
        Original class labels (n_samples,), can have K ≥ 2 unique values
    feature_types : list of str, optional
        'continuous' or 'categorical' for each feature

    Returns
    -------
    y_binary : np.ndarray
        Binary labels (0 or 1) for each sample
    group_A : set
        Set of original class labels assigned to super-class 0
    group_B : set
        Set of original class labels assigned to super-class 1
    """
    classes = np.unique(y)
    K = len(classes)

    # Trivial case: already binary
    if K == 2:
        # Map to 0 and 1
        label_map = {classes[0]: 0, classes[1]: 1}
        y_binary = np.array([label_map[label] for label in y])
        return y_binary, {classes[0]}, {classes[1]}

    # Edge case: only one class (pure node)
    if K == 1:
        return np.zeros(len(y), dtype=int), set(classes), set()

    n_features = X.shape[1]

    # Infer feature types if not provided
    if feature_types is None:
        feature_types = []
        for j in range(n_features):
            if is_categorical(X[:, j]):
                feature_types.append('categorical')
            else:
                feature_types.append('continuous')

    # =================================================================
    # Step 1: Compute class centroids
    # =================================================================
    # For continuous features: mean
    # For categorical features: we use one-hot encoding proportions

    # First, identify continuous features for centroid computation
    continuous_indices = [j for j in range(n_features)
                         if feature_types[j] == 'continuous']

    if len(continuous_indices) == 0:
        # All categorical: use chi-square distances or random split
        # Fall back to splitting classes by their sample counts
        class_counts = [(c, np.sum(y == c)) for c in classes]
        class_counts.sort(key=lambda x: x[1], reverse=True)

        # Split into two groups of roughly equal size
        total = sum(count for _, count in class_counts)
        cumsum = 0
        group_A = set()
        group_B = set()

        for c, count in class_counts:
            if cumsum < total / 2:
                group_A.add(c)
                cumsum += count
            else:
                group_B.add(c)

        # Ensure both groups are non-empty
        if len(group_B) == 0:
            group_B.add(group_A.pop())

        y_binary = np.array([0 if label in group_A else 1 for label in y])
        return y_binary, group_A, group_B

    # Compute centroids using continuous features
    centroids = np.zeros((K, len(continuous_indices)))

    for k_idx, c in enumerate(classes):
        mask = (y == c)
        class_data = X[mask][:, continuous_indices].astype(float)
        centroids[k_idx] = np.mean(class_data, axis=0)

    # =================================================================
    # Step 2: Apply 2-means clustering to centroids
    # =================================================================
    # Handle degenerate cases
    if K == 2:
        # Already binary
        group_A = {classes[0]}
        group_B = {classes[1]}
    else:
        # Use scipy's kmeans2 for 2-means clustering on centroids
        # Note: We cluster the K centroids, not the n samples
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                # Run 2-means multiple times to get stable result
                best_labels = None
                best_inertia = np.inf

                for seed in range(5):
                    np.random.seed(seed)
                    try:
                        _, labels = kmeans2(centroids, 2, iter=20, minit='points')

                        # Compute inertia (sum of squared distances to centroids)
                        cluster_centers = np.array([
                            np.mean(centroids[labels == 0], axis=0),
                            np.mean(centroids[labels == 1], axis=0)
                        ])

                        inertia = 0
                        for i, lab in enumerate(labels):
                            inertia += np.sum((centroids[i] - cluster_centers[lab])**2)

                        if inertia < best_inertia:
                            best_inertia = inertia
                            best_labels = labels
                    except Exception:
                        continue

                if best_labels is None:
                    # Fallback: split by first principal component
                    pca_direction = np.mean(centroids, axis=0)
                    projections = centroids @ pca_direction
                    median_proj = np.median(projections)
                    best_labels = (projections > median_proj).astype(int)

                labels = best_labels

        except Exception:
            # Fallback: random split
            labels = np.zeros(K, dtype=int)
            labels[K // 2:] = 1

        # Build group sets
        group_A = set(classes[labels == 0])
        group_B = set(classes[labels == 1])

        # Ensure both groups are non-empty
        if len(group_A) == 0:
            group_A.add(group_B.pop())
        if len(group_B) == 0:
            group_B.add(group_A.pop())

    # =================================================================
    # Step 3: Create binary labels for all samples
    # =================================================================
    y_binary = np.array([0 if label in group_A else 1 for label in y])

    return y_binary, group_A, group_B

In [3]:
def calc_anova_p_value(feature_values, y):
    """
    Calculate the p-value from a one-way ANOVA F-test.

    This tests whether the means of a continuous variable differ
    significantly between the two classes.

    Mathematical formulation:
    - H0: μ_1 = μ_2 (class means are equal)
    - H1: μ_1 ≠ μ_2 (class means differ)

    F-statistic = (Between-Group SS / df_between) / (Within-Group SS / df_within)

    Parameters
    ----------
    feature_values : np.ndarray
        Values of the continuous feature (n_samples,)
    y : np.ndarray
        Class labels, assumed to be 0 or 1 (n_samples,)

    Returns
    -------
    float
        The p-value from the F-test
    """
    # Get unique classes
    classes = np.unique(y)
    k = len(classes)  # Number of groups
    n = len(y)        # Total sample size

    # Compute overall mean
    grand_mean = np.mean(feature_values)

    # Compute group statistics
    group_means = []
    group_sizes = []

    for c in classes:
        mask = (y == c)
        group_values = feature_values[mask]
        group_means.append(np.mean(group_values))
        group_sizes.append(len(group_values))

    # Between-Group Sum of Squares (SSB)
    # SSB = Σ n_j * (x̄_j - x̄)²
    ss_between = 0.0
    for j in range(k):
        ss_between += group_sizes[j] * (group_means[j] - grand_mean) ** 2

    # Within-Group Sum of Squares (SSW)
    # SSW = Σ Σ (x_ij - x̄_j)²
    ss_within = 0.0
    for j, c in enumerate(classes):
        mask = (y == c)
        group_values = feature_values[mask]
        ss_within += np.sum((group_values - group_means[j]) ** 2)

    # Degrees of freedom
    df_between = k - 1
    df_within = n - k

    # Handle edge cases
    if df_within <= 0 or ss_within == 0:
        return 1.0  # Cannot compute F-statistic

    # Mean squares
    ms_between = ss_between / df_between
    ms_within = ss_within / df_within

    # F-statistic
    f_statistic = ms_between / ms_within

    # P-value from F-distribution
    p_value = 1.0 - stats.f.cdf(f_statistic, df_between, df_within)

    return p_value


In [4]:
def calc_chisquare_p_value(feature_values, y):
    """
    Calculate the p-value from a Chi-square test of independence.

    This tests whether a categorical variable is independent of
    the class label.

    Mathematical formulation:
    - H0: Variable and class are independent
    - H1: Variable and class are dependent

    χ² = Σ Σ (O_ij - E_ij)² / E_ij

    where:
    - O_ij = observed frequency
    - E_ij = expected frequency = (row_total * col_total) / grand_total

    Parameters
    ----------
    feature_values : np.ndarray
        Values of the categorical feature (n_samples,)
    y : np.ndarray
        Class labels (n_samples,)

    Returns
    -------
    float
        The p-value from the Chi-square test
    """
    # Get unique categories and classes
    categories = np.unique(feature_values)
    classes = np.unique(y)

    n_categories = len(categories)
    n_classes = len(classes)
    n_total = len(y)

    # Build contingency table manually
    # Rows = categories, Columns = classes
    observed = np.zeros((n_categories, n_classes))

    for i, cat in enumerate(categories):
        for j, cls in enumerate(classes):
            observed[i, j] = np.sum((feature_values == cat) & (y == cls))

    # Compute row and column totals
    row_totals = np.sum(observed, axis=1)
    col_totals = np.sum(observed, axis=0)

    # Compute expected frequencies
    # E_ij = (row_i_total * col_j_total) / grand_total
    expected = np.outer(row_totals, col_totals) / n_total

    # Handle cells with zero expected frequency
    # Add small epsilon to avoid division by zero
    epsilon = 1e-10
    expected = np.maximum(expected, epsilon)

    # Chi-square statistic
    chi_sq = np.sum((observed - expected) ** 2 / expected)

    # Degrees of freedom
    df = (n_categories - 1) * (n_classes - 1)

    if df <= 0:
        return 1.0  # Cannot compute

    # P-value from Chi-square distribution
    p_value = 1.0 - stats.chi2.cdf(chi_sq, df)

    return p_value


In [5]:
def calc_levene_p_value(feature_values, y):
    """
    Calculate the p-value from Levene's test for equality of variances.

    Levene's test is used as a backup when no variable passes the
    initial ANOVA/Chi-square tests. It detects differences in
    variance (spread) rather than mean (location).

    Mathematical formulation:
    1. Compute absolute deviations: z_ij = |x_ij - median(x_j)|
    2. Apply ANOVA to the z values

    This is the Brown-Forsythe variant using group medians,
    which is more robust to non-normality.

    Parameters
    ----------
    feature_values : np.ndarray
        Values of the continuous feature (n_samples,)
    y : np.ndarray
        Class labels (n_samples,)

    Returns
    -------
    float
        The p-value from Levene's test
    """
    classes = np.unique(y)
    k = len(classes)
    n = len(y)

    # Compute absolute deviations from group medians
    # z_ij = |x_ij - median(x_j)|
    z_values = np.zeros_like(feature_values, dtype=float)

    for c in classes:
        mask = (y == c)
        group_values = feature_values[mask]
        group_median = np.median(group_values)
        z_values[mask] = np.abs(group_values - group_median)

    # Now apply ANOVA to the z-values
    # This tests whether the average deviations differ between groups

    grand_mean_z = np.mean(z_values)

    group_means_z = []
    group_sizes = []

    for c in classes:
        mask = (y == c)
        group_z = z_values[mask]
        group_means_z.append(np.mean(group_z))
        group_sizes.append(len(group_z))

    # Between-Group SS for z-values
    ss_between = 0.0
    for j in range(k):
        ss_between += group_sizes[j] * (group_means_z[j] - grand_mean_z) ** 2

    # Within-Group SS for z-values
    ss_within = 0.0
    for j, c in enumerate(classes):
        mask = (y == c)
        group_z = z_values[mask]
        ss_within += np.sum((group_z - group_means_z[j]) ** 2)

    # Degrees of freedom
    df_between = k - 1
    df_within = n - k

    if df_within <= 0 or ss_within == 0:
        return 1.0

    # F-statistic
    f_statistic = (ss_between / df_between) / (ss_within / df_within)

    # P-value
    p_value = 1.0 - stats.f.cdf(f_statistic, df_between, df_within)

    return p_value


In [6]:
def is_categorical(feature_values, max_unique_ratio=0.05):
    """
    Heuristic to determine if a feature is categorical.

    A feature is considered categorical if:
    1. It contains non-numeric strings, OR
    2. The number of unique values is small relative to sample size

    Parameters
    ----------
    feature_values : np.ndarray
        Values of the feature
    max_unique_ratio : float
        Maximum ratio of unique values to total samples

    Returns
    -------
    bool
        True if the feature appears to be categorical
    """
    # Check if values are strings (object dtype)
    if feature_values.dtype == object:
        return True

    # Check if all values are integers
    if np.issubdtype(feature_values.dtype, np.integer):
        n_unique = len(np.unique(feature_values))
        n_total = len(feature_values)
        # If very few unique values, treat as categorical
        if n_unique <= 10 or n_unique / n_total <= max_unique_ratio:
            return True

    return False


In [7]:
def select_best_variable(X, y, feature_types=None):
    """
    Select the best splitting variable using Algorithm 3 from QUEST.

    This implements the unbiased variable selection procedure:

    1. For each variable, compute an association p-value:
       - Continuous: ANOVA F-test (tests difference in means)
       - Categorical: Chi-square test (tests independence)

    2. Apply Bonferroni correction: α* = α / M (M = number of variables)

    3. If any p-value < α*, select the variable with smallest p-value

    4. If no variable passes, apply Levene's test to continuous variables
       (detects variance differences instead of mean differences)

    Parameters
    ----------
    X : np.ndarray
        Feature matrix (n_samples, n_features)
    y : np.ndarray
        Class labels (n_samples,)
    feature_types : list of str, optional
        List indicating 'continuous' or 'categorical' for each feature.
        If None, types are inferred automatically.

    Returns
    -------
    tuple
        (best_feature_index, feature_type)
        Returns (None, None) if no suitable variable is found
    """
    n_samples, n_features = X.shape

    # Infer feature types if not provided
    if feature_types is None:
        feature_types = []
        for j in range(n_features):
            if is_categorical(X[:, j]):
                feature_types.append('categorical')
            else:
                feature_types.append('continuous')

    # Bonferroni correction
    alpha_corrected = ALPHA / n_features

    # Phase 1: ANOVA / Chi-square tests
    p_values = []

    for j in range(n_features):
        feature_values = X[:, j]

        if feature_types[j] == 'continuous':
            # Convert to float for continuous features
            feature_values = feature_values.astype(float)
            p_val = calc_anova_p_value(feature_values, y)
        else:
            # Categorical: Chi-square test
            p_val = calc_chisquare_p_value(feature_values, y)

        p_values.append(p_val)

    p_values = np.array(p_values)

    # Check if any variable passes the Bonferroni threshold
    significant_mask = p_values < alpha_corrected

    if np.any(significant_mask):
        # Select the variable with smallest p-value
        best_idx = np.argmin(p_values)
        return best_idx, feature_types[best_idx]

    # Phase 2: Levene's test for continuous variables
    # This tests for differences in variance (spread) rather than mean
    levene_p_values = np.ones(n_features)  # Default to 1.0 (not significant)

    continuous_indices = [j for j in range(n_features)
                         if feature_types[j] == 'continuous']

    if len(continuous_indices) == 0:
        # No continuous variables; fall back to smallest chi-square p-value
        best_idx = np.argmin(p_values)
        return best_idx, feature_types[best_idx]

    for j in continuous_indices:
        feature_values = X[:, j].astype(float)
        levene_p_values[j] = calc_levene_p_value(feature_values, y)

    # Bonferroni correction for Levene tests
    n_continuous = len(continuous_indices)
    alpha_levene = ALPHA / n_continuous if n_continuous > 0 else ALPHA

    # Check if any Levene test is significant
    levene_significant = levene_p_values < alpha_levene

    if np.any(levene_significant):
        # Select from significant Levene tests
        best_idx = np.argmin(levene_p_values)
        return best_idx, feature_types[best_idx]

    # Phase 3: No significant tests - select variable with smallest
    # combined p-value (ANOVA/Chi-square)
    best_idx = np.argmin(p_values)
    return best_idx, feature_types[best_idx]


---
### Block B — Categorical Transformation (Algorithm 2: CRIMCOORDS)

When a categorical variable is selected for splitting, we need to convert it to a numeric value to find a split point. QUEST uses **CRIMCOORDS** (CRItical Mean COORDinateS) for this purpose.

#### Why CRIMCOORDS?

The naive approach of one-hot encoding leads to a high-dimensional problem. CRIMCOORDS provides an **optimal 1D projection** by:

1. One-hot encoding the categories → matrix V
2. Centering V (subtracting column means)
3. Computing SVD to handle rank deficiency
4. Applying Linear Discriminant Analysis (LDA) on the reduced space
5. Computing a score ξ for each category

#### Mathematical Details

Let C be the number of categories. The one-hot matrix V is n × C. After centering:

$$\tilde{V} = V - \mathbf{1}_n \bar{v}^T$$

SVD gives: $\tilde{V} = U \Sigma W^T$

The rank r = C - 1 (due to centering). We work in the reduced r-dimensional space.

LDA finds the direction that maximizes class separation:

$$a = S_W^{-1}(\bar{x}_1 - \bar{x}_2)$$

The category scores are then: $\xi = W_r a$


In [8]:
def get_crimcoords_mapping(categorical_column, y):
    """
    Compute CRIMCOORDS mapping for a categorical variable.

    CRIMCOORDS (CRItical Mean COORDinateS) transforms categorical
    values to numeric scores that optimally separate the two classes.

    Algorithm (from QUEST paper):
    1. One-hot encode categories → matrix V (n × C)
    2. Center V by subtracting column means
    3. Apply SVD: V = U Σ W^T
    4. Determine rank r (typically C-1 due to centering)
    5. Project to r-dimensional space
    6. Compute LDA direction in reduced space
    7. Map back to category scores

    Parameters
    ----------
    categorical_column : np.ndarray
        Categorical feature values (n_samples,)
    y : np.ndarray
        Class labels, assumed to be 0 and 1 (n_samples,)

    Returns
    -------
    dict
        Mapping from category value to numeric score
        {category_1: score_1, category_2: score_2, ...}
    """
    categories = np.unique(categorical_column)
    n_categories = len(categories)
    n_samples = len(categorical_column)

    # Edge case: only one category
    if n_categories == 1:
        return {categories[0]: 0.0}

    # Edge case: two categories - simple encoding
    if n_categories == 2:
        # Compute class-conditional proportions for optimal split
        class_0_mask = (y == 0)
        class_1_mask = (y == 1)

        cat_0, cat_1 = categories[0], categories[1]

        # Proportion of each category in each class
        p0_cat0 = np.sum((categorical_column == cat_0) & class_0_mask) / max(np.sum(class_0_mask), 1)
        p1_cat0 = np.sum((categorical_column == cat_0) & class_1_mask) / max(np.sum(class_1_mask), 1)

        # Score based on log-odds ratio (simplified)
        return {
            cat_0: 0.0,
            cat_1: 1.0
        }

    # =================================================================
    # Step 1: One-Hot Encoding → Matrix V
    # =================================================================
    # Create category index mapping
    cat_to_idx = {cat: idx for idx, cat in enumerate(categories)}

    # Build one-hot matrix V (n_samples × n_categories)
    V = np.zeros((n_samples, n_categories))
    for i, cat in enumerate(categorical_column):
        V[i, cat_to_idx[cat]] = 1.0

    # =================================================================
    # Step 2: Center V (subtract column means)
    # =================================================================
    col_means = np.mean(V, axis=0)
    V_centered = V - col_means

    # =================================================================
    # Step 3: SVD Decomposition
    # =================================================================
    # V_centered = U @ Sigma @ W^T
    # This handles potential rank deficiency gracefully
    try:
        U, singular_values, Wt = linalg.svd(V_centered, full_matrices=False)
        W = Wt.T  # W is now (n_categories × n_components)
    except linalg.LinAlgError:
        # SVD failed; fall back to simple encoding
        return {cat: float(i) for i, cat in enumerate(categories)}

    # =================================================================
    # Step 4: Determine effective rank r
    # =================================================================
    # Due to centering, one singular value should be ~0
    # Keep components with significant singular values
    tol = 1e-10 * singular_values[0] if len(singular_values) > 0 else 1e-10
    r = np.sum(singular_values > tol)

    if r == 0:
        # Degenerate case
        return {cat: float(i) for i, cat in enumerate(categories)}

    # Truncate to rank r
    U_r = U[:, :r]
    Sigma_r = np.diag(singular_values[:r])
    W_r = W[:, :r]

    # =================================================================
    # Step 5: Project data to reduced r-dimensional space
    # =================================================================
    # The projected data is: X_reduced = U_r @ Sigma_r
    X_reduced = U_r @ Sigma_r

    # =================================================================
    # Step 6: Linear Discriminant Analysis in reduced space
    # =================================================================
    # Compute class means in reduced space
    class_0_mask = (y == 0)
    class_1_mask = (y == 1)

    n_0 = np.sum(class_0_mask)
    n_1 = np.sum(class_1_mask)

    if n_0 == 0 or n_1 == 0:
        # Only one class present
        return {cat: float(i) for i, cat in enumerate(categories)}

    mean_0 = np.mean(X_reduced[class_0_mask], axis=0)
    mean_1 = np.mean(X_reduced[class_1_mask], axis=0)

    # Within-class scatter matrix S_W
    # S_W = S_0 + S_1 where S_j = Σ (x - mean_j)(x - mean_j)^T
    X_0_centered = X_reduced[class_0_mask] - mean_0
    X_1_centered = X_reduced[class_1_mask] - mean_1

    S_W = X_0_centered.T @ X_0_centered + X_1_centered.T @ X_1_centered

    # Add regularization for numerical stability
    S_W += np.eye(r) * 1e-6

    # LDA direction: a = S_W^{-1} (mean_1 - mean_0)
    mean_diff = mean_1 - mean_0

    try:
        a = linalg.solve(S_W, mean_diff)
    except linalg.LinAlgError:
        # Singular matrix; use pseudo-inverse
        a = linalg.lstsq(S_W, mean_diff)[0]

    # =================================================================
    # Step 7: Compute category scores ξ
    # =================================================================
    # Map the LDA direction back to category space
    # ξ = W_r @ a gives a score for each category
    xi = W_r @ a

    # Create the mapping dictionary
    crimcoords_map = {cat: float(xi[idx]) for idx, cat in enumerate(categories)}

    return crimcoords_map


---
### Block C — Split Point Calculation (Algorithm 1: QDA Split)

Once a variable is selected, QUEST uses **Quadratic Discriminant Analysis (QDA)** to find the optimal split point.

#### Why QDA?

Unlike exhaustive search (as in CART), QDA provides a **closed-form solution** for the split point, making QUEST computationally efficient.

#### Mathematical Formulation

Assume the selected variable follows different normal distributions in each class:

- Class 0: X ~ N(mu_0, sigma_0^2)
- Class 1: X ~ N(mu_1, sigma_1^2)

The QDA split point is where the posterior probabilities are equal. This leads to a quadratic equation:

A*t^2 + B*t + C = 0

The physically meaningful root (the one between the class means) is selected.

---
### Block D — Cost-Complexity Pruning

QUEST uses **post-pruning** via Minimal Cost-Complexity Pruning (same as CART).

The cost-complexity criterion for a subtree T is:

$$R_\alpha(T) = R(T) + \alpha |T|$$

where:
- $R(T)$ is the misclassification rate (empirical risk)
- $|T|$ is the number of terminal nodes
- $\alpha \geq 0$ is the complexity parameter

For each internal node t, we compute the effective alpha:

$$\alpha_{eff}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}$$

Nodes with smallest $\alpha_{eff}$ are pruned first, generating a sequence of subtrees.

In [9]:
def find_qda_split_point(values, y):
    """
    Find the optimal split point using Quadratic Discriminant Analysis.

    QDA assumes each class follows a normal distribution with potentially
    different variances. The split point is where the posterior probabilities
    are equal.

    Mathematical derivation:
    - Equating log posteriors leads to a quadratic equation in t
    - We solve: A*t^2 + B*t + C = 0
    - Select the root that lies between the class means

    Parameters
    ----------
    values : np.ndarray
        Numeric values of the splitting variable (n_samples,)
    y : np.ndarray
        Class labels, assumed to be 0 and 1 (n_samples,)

    Returns
    -------
    float
        The optimal split threshold
    """
    values = np.asarray(values, dtype=float)

    # Separate by class
    class_0_mask = (y == 0)
    class_1_mask = (y == 1)

    values_0 = values[class_0_mask]
    values_1 = values[class_1_mask]

    n_0 = len(values_0)
    n_1 = len(values_1)

    # Edge cases
    if n_0 == 0 or n_1 == 0:
        return np.median(values)

    # Compute class statistics
    mu_0 = np.mean(values_0)
    mu_1 = np.mean(values_1)

    # Variance with Bessel's correction (n-1)
    # Add small epsilon to prevent division by zero
    epsilon = 1e-10

    if n_0 > 1:
        var_0 = np.var(values_0, ddof=1)
    else:
        var_0 = epsilon

    if n_1 > 1:
        var_1 = np.var(values_1, ddof=1)
    else:
        var_1 = epsilon

    # Ensure positive variances
    var_0 = max(var_0, epsilon)
    var_1 = max(var_1, epsilon)

    sigma_0 = np.sqrt(var_0)
    sigma_1 = np.sqrt(var_1)

    # =================================================================
    # Special case: Equal variances -> Linear discriminant
    # =================================================================
    if np.abs(var_0 - var_1) < epsilon * max(var_0, var_1):
        # LDA solution: midpoint between means
        split_point = (mu_0 + mu_1) / 2.0
        return split_point

    # =================================================================
    # General case: Solve quadratic equation
    # =================================================================
    # QDA decision boundary:
    # A*t^2 + B*t + C = 0

    inv_var_0 = 1.0 / var_0
    inv_var_1 = 1.0 / var_1

    # Coefficients
    A = 0.5 * (inv_var_0 - inv_var_1)
    B = mu_1 * inv_var_1 - mu_0 * inv_var_0
    C = 0.5 * (mu_0**2 * inv_var_0 - mu_1**2 * inv_var_1) + np.log(sigma_1 / sigma_0)

    # Discriminant
    discriminant = B**2 - 4*A*C

    if discriminant < 0:
        # No real roots; fall back to midpoint between means
        return (mu_0 + mu_1) / 2.0

    if np.abs(A) < epsilon:
        # Linear case (shouldn't happen due to earlier check, but safety)
        if np.abs(B) < epsilon:
            return (mu_0 + mu_1) / 2.0
        return -C / B

    # Two roots
    sqrt_disc = np.sqrt(discriminant)
    root_1 = (-B + sqrt_disc) / (2 * A)
    root_2 = (-B - sqrt_disc) / (2 * A)

    # =================================================================
    # Select the physically meaningful root
    # =================================================================
    # The best split point should lie between the class means
    min_mean = min(mu_0, mu_1)
    max_mean = max(mu_0, mu_1)

    # Also consider the data range
    data_min = np.min(values)
    data_max = np.max(values)

    candidates = []

    # Prefer roots between the means
    if min_mean <= root_1 <= max_mean:
        candidates.append((0, root_1))  # Priority 0 = best
    elif data_min <= root_1 <= data_max:
        candidates.append((1, root_1))  # Priority 1 = within data

    if min_mean <= root_2 <= max_mean:
        candidates.append((0, root_2))
    elif data_min <= root_2 <= data_max:
        candidates.append((1, root_2))

    if len(candidates) > 0:
        # Select the candidate with best priority
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1]

    # Fallback: midpoint between means
    return (mu_0 + mu_1) / 2.0


In [10]:
def compute_node_impurity(node, n_total):
    """
    Compute the misclassification rate (impurity) for a node.

    For cost-complexity pruning, the impurity of node t is:
    R(t) = (n_t / n) * (1 - max_k(p_k))

    where:
    - n_t is the number of samples at node t
    - n is the total number of samples in the tree
    - p_k is the proportion of class k at node t

    Parameters
    ----------
    node : Node
        The tree node
    n_total : int
        Total number of samples in the training set

    Returns
    -------
    float
        The node impurity (contribution to overall misclassification rate)
    """
    if node.n_samples == 0:
        return 0.0

    # Proportion of samples at this node
    node_weight = node.n_samples / n_total

    # Find the majority class proportion
    if len(node.class_counts) == 0:
        max_proportion = 1.0
    else:
        max_count = max(node.class_counts.values())
        max_proportion = max_count / node.n_samples

    # Misclassification rate: 1 - (majority class proportion)
    misclass_rate = 1.0 - max_proportion

    # Weighted impurity
    return node_weight * misclass_rate


def compute_subtree_impurity(node, n_total):
    """
    Recursively compute the total impurity of the subtree rooted at node.

    For a leaf: R(T_t) = R(t)
    For an internal node: R(T_t) = R(T_left) + R(T_right)

    Also computes the number of leaves |T_t| in the subtree.

    Parameters
    ----------
    node : Node
        The tree node
    n_total : int
        Total number of samples in the training set

    Returns
    -------
    tuple (float, int)
        (subtree_impurity, n_leaves)
    """
    if node.is_leaf:
        impurity = compute_node_impurity(node, n_total)
        node.impurity = impurity
        node.subtree_impurity = impurity
        node.n_leaves = 1
        return impurity, 1

    # Recurse on children
    left_impurity, left_leaves = compute_subtree_impurity(node.left_child, n_total)
    right_impurity, right_leaves = compute_subtree_impurity(node.right_child, n_total)

    # Impurity if this node were a leaf
    node.impurity = compute_node_impurity(node, n_total)

    # Subtree impurity (sum of children)
    node.subtree_impurity = left_impurity + right_impurity
    node.n_leaves = left_leaves + right_leaves

    return node.subtree_impurity, node.n_leaves


def compute_effective_alpha(node):
    """
    Compute the effective alpha for a node (used in minimal cost-complexity pruning).

    The effective alpha is:
    α_eff(t) = (R(t) - R(T_t)) / (|T_t| - 1)

    This represents the "cost" of keeping the subtree vs. pruning to a leaf.
    Nodes with smaller α_eff are pruned first.

    Parameters
    ----------
    node : Node
        The tree node

    Returns
    -------
    float
        The effective alpha value
    """
    if node.is_leaf or node.n_leaves <= 1:
        return np.inf  # Cannot prune a leaf

    numerator = node.impurity - node.subtree_impurity
    denominator = node.n_leaves - 1

    if denominator <= 0:
        return np.inf

    return numerator / denominator


def find_min_alpha_node(node, min_alpha=np.inf, min_node=None):
    """
    Find the internal node with the smallest effective alpha.

    Parameters
    ----------
    node : Node
        Current node in traversal
    min_alpha : float
        Current minimum alpha found
    min_node : Node or None
        Node with current minimum alpha

    Returns
    -------
    tuple (float, Node)
        (min_alpha, min_node)
    """
    if node.is_leaf:
        return min_alpha, min_node

    # Compute effective alpha for this node
    alpha = compute_effective_alpha(node)

    if alpha < min_alpha:
        min_alpha = alpha
        min_node = node

    # Recurse on children
    min_alpha, min_node = find_min_alpha_node(node.left_child, min_alpha, min_node)
    min_alpha, min_node = find_min_alpha_node(node.right_child, min_alpha, min_node)

    return min_alpha, min_node


def prune_node(node):
    """
    Convert an internal node to a leaf (prune its subtree).

    Parameters
    ----------
    node : Node
        The node to prune
    """
    node.is_leaf = True
    node.left_child = None
    node.right_child = None
    node.n_leaves = 1
    node.subtree_impurity = node.impurity

    # Set prediction to majority class
    if len(node.class_counts) > 0:
        node.prediction = max(node.class_counts.keys(),
                             key=lambda k: node.class_counts[k])


def cost_complexity_prune(root, ccp_alpha, n_total):
    """
    Apply cost-complexity pruning to the tree.

    This implements the minimal cost-complexity pruning algorithm:
    1. Compute impurities for all nodes
    2. Find node with smallest effective alpha
    3. If α_eff <= ccp_alpha, prune and repeat
    4. Stop when no node can be pruned

    Parameters
    ----------
    root : Node
        The root of the tree
    ccp_alpha : float
        The complexity parameter (α)
        - α = 0: no pruning
        - larger α: more aggressive pruning
    n_total : int
        Total number of training samples

    Returns
    -------
    Node
        The pruned tree root
    list
        List of (alpha, n_leaves) for the pruning path
    """
    if ccp_alpha < 0:
        raise ValueError("ccp_alpha must be non-negative")

    pruning_path = []

    # Initial computation of all impurities
    compute_subtree_impurity(root, n_total)
    pruning_path.append((0.0, root.n_leaves))

    if ccp_alpha == 0:
        return root, pruning_path

    # Iteratively prune until no beneficial pruning remains
    while True:
        # Recompute impurities after any structural changes
        compute_subtree_impurity(root, n_total)

        # Find node with minimum effective alpha
        min_alpha, min_node = find_min_alpha_node(root)

        if min_node is None or min_alpha > ccp_alpha:
            # No more nodes to prune at this alpha level
            break

        # Prune this node
        prune_node(min_node)

        # Recompute and record
        compute_subtree_impurity(root, n_total)
        pruning_path.append((min_alpha, root.n_leaves))

        # Check if tree is now just the root
        if root.is_leaf:
            break

    return root, pruning_path


def generate_pruning_path(root, n_total):
    """
    Generate the complete pruning path (sequence of subtrees).

    This creates a sequence of (alpha, tree) pairs where increasing
    alpha produces smaller trees.

    Parameters
    ----------
    root : Node
        The root of the fully grown tree
    n_total : int
        Total number of training samples

    Returns
    -------
    list of tuples
        [(alpha_0, n_leaves_0), (alpha_1, n_leaves_1), ...]
        Sorted by increasing alpha
    """
    import copy

    # Work on a copy to preserve original tree
    path = []

    # Initial state
    compute_subtree_impurity(root, n_total)
    path.append((0.0, root.n_leaves, copy.deepcopy(root)))

    # Keep pruning until only root remains
    current_alpha = 0.0

    while not root.is_leaf:
        compute_subtree_impurity(root, n_total)
        min_alpha, min_node = find_min_alpha_node(root)

        if min_node is None:
            break

        prune_node(min_node)
        compute_subtree_impurity(root, n_total)

        path.append((min_alpha, root.n_leaves, copy.deepcopy(root)))

    return path

---
## 3. Tree Structure

Now we define the tree data structure and the main `QuestTree` class.


In [11]:
class Node:
    """
    A node in the QUEST decision tree.

    Extended to support:
    - Multiclass classification (via super-class clustering)
    - Cost-complexity pruning metadata

    Attributes
    ----------
    feature_index : int or None
        Index of the splitting feature (None for leaf nodes)
    threshold : float or None
        Split threshold for continuous features (or CRIMCOORDS-transformed)
    crimcoords_map : dict or None
        Mapping from category to numeric score (for categorical splits)
    left_child : Node or None
        Left child node (values <= threshold go left)
    right_child : Node or None
        Right child node (values > threshold go right)
    is_leaf : bool
        Whether this is a leaf node
    prediction : int or None
        Class prediction for leaf nodes (original label, not binary)
    depth : int
        Depth of this node in the tree
    n_samples : int
        Number of samples at this node
    class_counts : dict
        Count of samples for each class at this node {class_label: count}
    superclass_groups : tuple or None
        (group_A, group_B) sets of original class labels used for this split
    impurity : float
        Misclassification rate at this node (for pruning)
    subtree_impurity : float
        Total weighted misclassification rate of subtree rooted here
    n_leaves : int
        Number of leaf nodes in subtree rooted here
    """

    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.crimcoords_map = None
        self.left_child = None
        self.right_child = None
        self.is_leaf = False
        self.prediction = None
        self.depth = 0
        self.n_samples = 0
        self.class_counts = {}
        self.superclass_groups = None

        # Pruning-related attributes
        self.impurity = 0.0  # R(t): misclassification rate if this were a leaf
        self.subtree_impurity = 0.0  # R(T_t): impurity of subtree
        self.n_leaves = 1  # |T_t|: number of leaves in subtree

    def __repr__(self):
        if self.is_leaf:
            return f"Leaf(pred={self.prediction}, n={self.n_samples})"
        else:
            return f"Node(feature={self.feature_index}, threshold={self.threshold:.4f}, n={self.n_samples})"

In [12]:
class QuestTree:
    """
    QUEST Decision Tree Classifier.

    Full implementation of the QUEST algorithm supporting:
    - Multiclass classification via Super-Class Clustering (Algorithm 4)
    - Unbiased variable selection via statistical tests (Algorithm 3)
    - CRIMCOORDS for categorical variable transformation (Algorithm 2)
    - QDA-based split point selection (Algorithm 1)
    - Cost-complexity post-pruning (CART-style)

    Parameters
    ----------
    max_depth : int
        Maximum depth of the tree (safety limit; pruning is primary regularization)
    min_samples_split : int
        Minimum samples required to attempt a split
    min_samples_leaf : int
        Minimum samples required in each leaf
    ccp_alpha : float
        Cost-complexity pruning parameter (α ≥ 0)
        - α = 0: no pruning (default)
        - larger α: more aggressive pruning
    feature_types : list of str, optional
        List indicating 'continuous' or 'categorical' for each feature

    Attributes
    ----------
    root_ : Node
        The root node of the fitted tree
    n_features_ : int
        Number of features in the training data
    classes_ : np.ndarray
        Unique class labels (supports K ≥ 2)
    n_samples_ : int
        Number of training samples
    pruning_path_ : list
        Sequence of (alpha, n_leaves) from pruning
    """

    def __init__(self, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT,
                 min_samples_leaf=MIN_SAMPLES_LEAF, ccp_alpha=DEFAULT_CCP_ALPHA,
                 feature_types=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.ccp_alpha = ccp_alpha
        self.feature_types = feature_types

        self.root_ = None
        self.n_features_ = None
        self.classes_ = None
        self.n_samples_ = None
        self.pruning_path_ = None

    def fit(self, X, y):
        """
        Build the QUEST decision tree from training data.

        The algorithm:
        1. Grow a large tree (minimal stopping criteria)
        2. Apply cost-complexity pruning to find optimal subtree

        Parameters
        ----------
        X : np.ndarray
            Training feature matrix (n_samples, n_features)
        y : np.ndarray
            Training labels (n_samples,), supports K ≥ 2 classes

        Returns
        -------
        self
        """
        X = np.asarray(X)
        y = np.asarray(y)

        self.n_features_ = X.shape[1]
        self.n_samples_ = X.shape[0]
        self.classes_ = np.unique(y)

        # QUEST supports K ≥ 2 classes via super-class clustering
        if len(self.classes_) < 2:
            raise ValueError("Need at least 2 classes for classification. "
                           f"Found {len(self.classes_)} class(es).")

        # Infer feature types if not provided
        if self.feature_types is None:
            self.feature_types = []
            for j in range(self.n_features_):
                if is_categorical(X[:, j]):
                    self.feature_types.append('categorical')
                else:
                    self.feature_types.append('continuous')

        # =============================================================
        # Phase 1: Grow a large tree (minimal early stopping)
        # =============================================================
        self.root_ = self._grow_tree(X, y, depth=0)

        # =============================================================
        # Phase 2: Cost-Complexity Pruning
        # =============================================================
        if self.ccp_alpha > 0:
            self.root_, self.pruning_path_ = cost_complexity_prune(
                self.root_, self.ccp_alpha, self.n_samples_
            )
        else:
            # Still compute impurities for analysis
            compute_subtree_impurity(self.root_, self.n_samples_)
            self.pruning_path_ = [(0.0, self.root_.n_leaves)]

        return self

    def _grow_tree(self, X, y, depth):
        """
        Recursively grow the decision tree.

        This implements the core QUEST algorithm:
        1. Check stopping conditions
        2. Apply Super-Class Clustering if K > 2 (Algorithm 4)
        3. Select best variable (Algorithm 3)
        4. Apply CRIMCOORDS if categorical (Algorithm 2)
        5. Find split point using QDA (Algorithm 1)
        6. Create node and recurse

        Parameters
        ----------
        X : np.ndarray
            Feature matrix for this node
        y : np.ndarray
            Original class labels (can have K ≥ 2 unique values)
        depth : int
            Current depth in the tree

        Returns
        -------
        Node
            The constructed tree node
        """
        n_samples = len(y)
        classes_at_node = np.unique(y)
        n_classes = len(classes_at_node)

        node = Node()
        node.depth = depth
        node.n_samples = n_samples

        # Store class counts for pruning and prediction
        node.class_counts = {c: int(np.sum(y == c)) for c in classes_at_node}

        # =============================================================
        # Step 1: Check stopping conditions
        # =============================================================

        # (a) Pure node: all samples belong to one class
        if n_classes == 1:
            node.is_leaf = True
            node.prediction = classes_at_node[0]
            return node

        # (b) Maximum depth reached
        if depth >= self.max_depth:
            node.is_leaf = True
            node.prediction = self._majority_class(y)
            return node

        # (c) Too few samples to split
        if n_samples < self.min_samples_split:
            node.is_leaf = True
            node.prediction = self._majority_class(y)
            return node

        # (d) Cannot create leaves with minimum samples
        if n_samples < 2 * self.min_samples_leaf:
            node.is_leaf = True
            node.prediction = self._majority_class(y)
            return node

        # =============================================================
        # Step 2: Super-Class Clustering (Algorithm 4)
        # Convert K-class problem to binary for QDA/CRIMCOORDS
        # =============================================================
        y_binary, group_A, group_B = compute_superclass_labels(
            X, y, self.feature_types
        )
        node.superclass_groups = (group_A, group_B)

        # =============================================================
        # Step 3: Variable Selection (Algorithm 3)
        # Use binary labels for unbiased selection
        # =============================================================
        best_feature_idx, feature_type = select_best_variable(
            X, y_binary, self.feature_types
        )

        if best_feature_idx is None:
            # No suitable variable found
            node.is_leaf = True
            node.prediction = self._majority_class(y)
            return node

        node.feature_index = best_feature_idx

        # Get the values of the selected feature
        feature_values = X[:, best_feature_idx]

        # =============================================================
        # Step 4: CRIMCOORDS if categorical (Algorithm 2)
        # Use binary labels for optimal projection
        # =============================================================
        if feature_type == 'categorical':
            # Transform categorical to numeric using CRIMCOORDS
            crimcoords_map = get_crimcoords_mapping(feature_values, y_binary)
            node.crimcoords_map = crimcoords_map

            # Apply transformation
            transformed_values = np.array([
                crimcoords_map.get(val, 0.0) for val in feature_values
            ])
        else:
            # Continuous feature: use directly
            transformed_values = feature_values.astype(float)
            node.crimcoords_map = None

        # =============================================================
        # Step 5: Find split point using QDA (Algorithm 1)
        # Use binary labels for the quadratic discriminant
        # =============================================================
        threshold = find_qda_split_point(transformed_values, y_binary)
        node.threshold = threshold

        # =============================================================
        # Step 6: Split the data
        # =============================================================
        left_mask = transformed_values <= threshold
        right_mask = ~left_mask

        # Check for empty or too-small splits
        n_left = np.sum(left_mask)
        n_right = np.sum(right_mask)

        if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
            node.is_leaf = True
            node.prediction = self._majority_class(y)
            return node

        # =============================================================
        # Step 7: Recursive calls with ORIGINAL labels
        # The super-class grouping is node-specific; children use original y
        # =============================================================
        node.left_child = self._grow_tree(
            X[left_mask], y[left_mask], depth + 1
        )
        node.right_child = self._grow_tree(
            X[right_mask], y[right_mask], depth + 1
        )

        return node

    def _majority_class(self, y):
        """
        Return the majority class in y.

        Parameters
        ----------
        y : np.ndarray
            Class labels (any unique values)

        Returns
        -------
            The majority class label
        """
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]

    def predict_row(self, row):
        """
        Predict the class for a single sample.

        Parameters
        ----------
        row : np.ndarray
            Feature values for one sample (n_features,)

        Returns
        -------
            Predicted class label
        """
        node = self.root_

        while not node.is_leaf:
            feature_value = row[node.feature_index]

            # Transform if categorical
            if node.crimcoords_map is not None:
                # Look up the CRIMCOORDS score
                transformed_value = node.crimcoords_map.get(feature_value, 0.0)
            else:
                transformed_value = float(feature_value)

            # Traverse left or right
            if transformed_value <= node.threshold:
                node = node.left_child
            else:
                node = node.right_child

        return node.prediction

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Parameters
        ----------
        X : np.ndarray
            Feature matrix (n_samples, n_features)

        Returns
        -------
        np.ndarray
            Predicted class labels (n_samples,)
        """
        X = np.asarray(X)
        predictions = [self.predict_row(row) for row in X]
        return np.array(predictions)

    def get_n_leaves(self):
        """Return the number of leaves in the tree."""
        if self.root_ is None:
            return 0
        return self.root_.n_leaves

    def get_depth(self):
        """Return the maximum depth of the tree."""
        if self.root_ is None:
            return 0
        return self._get_max_depth(self.root_)

    def _get_max_depth(self, node):
        """Recursively compute maximum depth."""
        if node.is_leaf:
            return node.depth
        return max(
            self._get_max_depth(node.left_child),
            self._get_max_depth(node.right_child)
        )

    def print_tree(self, node=None, indent=""):
        """
        Print a text representation of the tree.

        Parameters
        ----------
        node : Node, optional
            Starting node (default: root)
        indent : str
            Indentation string for formatting
        """
        if node is None:
            node = self.root_
            print("QUEST Decision Tree Structure:")
            print(f"  Classes: {list(self.classes_)}")
            print(f"  Leaves: {self.get_n_leaves()}")
            print(f"  Depth: {self.get_depth()}")
            print("=" * 60)

        if node.is_leaf:
            class_dist = ", ".join([f"{k}:{v}" for k, v in node.class_counts.items()])
            print(f"{indent}[LEAF] Predict: {node.prediction} | n={node.n_samples} | dist=({class_dist})")
        else:
            feature_type = self.feature_types[node.feature_index]

            # Show super-class grouping for multiclass
            if len(self.classes_) > 2 and node.superclass_groups is not None:
                group_A, group_B = node.superclass_groups
                print(f"{indent}[Node] Feature {node.feature_index} ({feature_type})")
                print(f"{indent}       Super-classes: {list(group_A)} vs {list(group_B)}")
            else:
                print(f"{indent}[Node] Feature {node.feature_index} ({feature_type})")

            if node.crimcoords_map is not None:
                print(f"{indent}       CRIMCOORDS threshold: {node.threshold:.4f}")
            else:
                print(f"{indent}       Threshold: {node.threshold:.4f}")
            print(f"{indent}       n_samples: {node.n_samples}")

            print(f"{indent}  |-- Left (<=):")
            self.print_tree(node.left_child, indent + "  |   ")

            print(f"{indent}  |-- Right (>):")
            self.print_tree(node.right_child, indent + "      ")

---
## 4. Testing & Demonstration

Let's test our QUEST implementation on synthetic and real datasets:
1. **Binary classification** - to verify backward compatibility
2. **Multiclass classification (K=4)** - to validate super-class clustering
3. **Cost-complexity pruning** - to demonstrate post-pruning
4. **Iris Dataset**
5. **UCI Car Evaluation**

In [13]:
# =============================================================================
# Create a synthetic dataset
# =============================================================================

np.random.seed(42)

n_samples = 200

# Generate class labels
y = np.array([0] * 100 + [1] * 100)

# Feature 1: Continuous - different means for each class
feature_1_class_0 = np.random.normal(loc=2.0, scale=1.0, size=100)
feature_1_class_1 = np.random.normal(loc=5.0, scale=1.5, size=100)
feature_1 = np.concatenate([feature_1_class_0, feature_1_class_1])

# Feature 2: Continuous - different variances (for Levene's test)
feature_2_class_0 = np.random.normal(loc=0.0, scale=0.5, size=100)
feature_2_class_1 = np.random.normal(loc=0.0, scale=2.0, size=100)
feature_2 = np.concatenate([feature_2_class_0, feature_2_class_1])

# Feature 3: Categorical - correlated with class
categories = ['A', 'B', 'C', 'D']
# Class 0 tends to have A or B
feature_3_class_0 = np.random.choice(['A', 'B', 'C', 'D'], size=100, p=[0.4, 0.4, 0.1, 0.1])
# Class 1 tends to have C or D
feature_3_class_1 = np.random.choice(['A', 'B', 'C', 'D'], size=100, p=[0.1, 0.1, 0.4, 0.4])
feature_3 = np.concatenate([feature_3_class_0, feature_3_class_1])

# Feature 4: Noise (continuous, unrelated to class)
feature_4 = np.random.normal(loc=0.0, scale=1.0, size=n_samples)

# Combine into feature matrix
# Note: We use object dtype to accommodate mixed types
X = np.column_stack([feature_1, feature_2, feature_3, feature_4])

# Shuffle the data
shuffle_idx = np.random.permutation(n_samples)
X = X[shuffle_idx]
y = y[shuffle_idx]

print("Synthetic Dataset Created:")
print(f"  - n_samples: {n_samples}")
print(f"  - n_features: 4")
print(f"  - Feature 1: Continuous (different means)")
print(f"  - Feature 2: Continuous (different variances)")
print(f"  - Feature 3: Categorical (4 categories)")
print(f"  - Feature 4: Noise (irrelevant)")
print(f"\nFirst 5 rows:")
for i in range(5):
    print(f"  X[{i}] = [{float(X[i, 0]):6.2f}, {float(X[i, 1]):6.2f}, '{X[i, 2]}', {float(X[i, 3]):6.2f}]  y = {y[i]}")


Synthetic Dataset Created:
  - n_samples: 200
  - n_features: 4
  - Feature 1: Continuous (different means)
  - Feature 2: Continuous (different variances)
  - Feature 3: Categorical (4 categories)
  - Feature 4: Noise (irrelevant)

First 5 rows:
  X[0] = [  1.61,   0.46, 'D',   1.88]  y = 0
  X[1] = [  4.63,   0.20, 'B',  -1.35]  y = 1
  X[2] = [  8.29,  -2.43, 'C',   1.27]  y = 1
  X[3] = [  2.38,  -0.54, 'B',  -0.42]  y = 0
  X[4] = [  2.50,   0.18, 'B',   0.34]  y = 0


In [14]:
# =============================================================================
# Train the QUEST Tree
# =============================================================================

# Specify feature types explicitly
feature_types = ['continuous', 'continuous', 'categorical', 'continuous']

# Create and fit the tree
tree = QuestTree(
    max_depth=5,
    min_samples_split=10,
    feature_types=feature_types
)

print("Training QUEST Decision Tree...")
tree.fit(X, y)
print("Training complete!\n")

# Print tree structure
tree.print_tree()


Training QUEST Decision Tree...
Training complete!

QUEST Decision Tree Structure:
  Classes: [np.int64(0), np.int64(1)]
  Leaves: 11
  Depth: 5
[Node] Feature 0 (continuous)
       Threshold: 2.9207
       n_samples: 200
  |-- Left (<=):
  |   [Node] Feature 2 (categorical)
  |          CRIMCOORDS threshold: -0.0027
  |          n_samples: 92
  |     |-- Left (<=):
  |     |   [Node] Feature 0 (continuous)
  |     |          Threshold: 2.1266
  |     |          n_samples: 72
  |     |     |-- Left (<=):
  |     |     |   [LEAF] Predict: 0 | n=49 | dist=(0:49)
  |     |     |-- Right (>):
  |     |         [Node] Feature 2 (categorical)
  |     |                CRIMCOORDS threshold: 0.7727
  |     |                n_samples: 23
  |     |           |-- Left (<=):
  |     |           |   [LEAF] Predict: 0 | n=10 | dist=(0:10)
  |     |           |-- Right (>):
  |     |               [Node] Feature 1 (continuous)
  |     |                      Threshold: 0.3582
  |     |                 

In [15]:
# =============================================================================
# Make Predictions and Evaluate
# =============================================================================

# Predict on training data
y_pred = tree.predict(X)

# Calculate accuracy
accuracy = np.mean(y_pred == y)

print(f"Training Accuracy: {accuracy:.2%}")
print(f"\nSample Predictions:")
print(f"{'Index':<8} {'Actual':<10} {'Predicted':<10} {'Correct':<10}")
print("-" * 40)
for i in range(10):
    correct = "Y" if y[i] == y_pred[i] else "X"
    print(f"{i:<8} {y[i]:<10} {y_pred[i]:<10} {correct:<10}")


Training Accuracy: 93.00%

Sample Predictions:
Index    Actual     Predicted  Correct   
----------------------------------------
0        0          0          Y         
1        1          1          Y         
2        1          1          Y         
3        0          0          Y         
4        0          0          Y         
5        0          0          Y         
6        0          0          Y         
7        1          1          Y         
8        0          0          Y         
9        1          1          Y         


### TEST 1: Binary Classification (Backward Compatibility)

In [16]:
# ============================================================
# TEST 1: Binary Classification (Backward Compatibility)
# ============================================================
print("=" * 60)
print("TEST 1: Binary Classification")
print("=" * 60)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Generate binary classification dataset
X_bin, y_bin = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    random_state=42,
    class_sep=1.5
)

# Define feature types (all continuous)
feature_types_bin = ['continuous'] * 6

# Split data
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.3, random_state=42
)

# Train QUEST tree
tree_bin = QuestTree(
    max_depth=4,
    min_samples_split=10,
    feature_types=feature_types_bin
)
tree_bin.fit(X_train_bin, y_train_bin)

# Predictions
y_pred_bin = tree_bin.predict(X_test_bin)

print(f"\nClasses detected: {tree_bin.classes_}")
print(f"Training samples: {len(X_train_bin)}")
print(f"Test samples: {len(X_test_bin)}")
print(f"\nAccuracy: {accuracy_score(y_test_bin, y_pred_bin):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_bin, y_pred_bin))
print("\nTree Structure:")
tree_bin.print_tree()

TEST 1: Binary Classification

Classes detected: [0 1]
Training samples: 210
Test samples: 90

Accuracy: 0.8889

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.86      0.89        49
           1       0.84      0.93      0.88        41

    accuracy                           0.89        90
   macro avg       0.89      0.89      0.89        90
weighted avg       0.89      0.89      0.89        90


Tree Structure:
QUEST Decision Tree Structure:
  Classes: [np.int64(0), np.int64(1)]
  Leaves: 10
  Depth: 4
[Node] Feature 5 (continuous)
       Threshold: -0.7754
       n_samples: 210
  |-- Left (<=):
  |   [Node] Feature 0 (continuous)
  |          Threshold: 0.5241
  |          n_samples: 126
  |     |-- Left (<=):
  |     |   [LEAF] Predict: 0 | n=53 | dist=(0:53)
  |     |-- Right (>):
  |         [Node] Feature 1 (continuous)
  |                Threshold: -0.5289
  |                n_samples: 73
  |           |-- Left (<=)

### TEST 2: Multiclass Classification (K=4 classes)

In [17]:
# ============================================================
# TEST 2: Multiclass Classification (K=4 classes)
# ============================================================
print("=" * 60)
print("TEST 2: Multiclass Classification (K=4)")
print("=" * 60)

# Generate 4-class classification dataset
X_multi, y_multi = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=6,
    n_redundant=1,
    n_classes=4,
    n_clusters_per_class=1,
    random_state=42,
    class_sep=1.2
)

# Define feature types (all continuous)
feature_types_multi = ['continuous'] * 8

# Split data
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42
)

# Train QUEST tree with super-class clustering
tree_multi = QuestTree(
    max_depth=5,
    min_samples_split=15,
    feature_types=feature_types_multi
)
tree_multi.fit(X_train_multi, y_train_multi)

# Predictions
y_pred_multi = tree_multi.predict(X_test_multi)

print(f"\nClasses detected: {tree_multi.classes_}")
print(f"Training samples: {len(X_train_multi)}")
print(f"Test samples: {len(X_test_multi)}")
print(f"\nAccuracy: {accuracy_score(y_test_multi, y_pred_multi):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_multi, y_pred_multi))
print("\nTree Structure:")
tree_multi.print_tree()

TEST 2: Multiclass Classification (K=4)

Classes detected: [0 1 2 3]
Training samples: 350
Test samples: 150

Accuracy: 0.6000

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.75      0.69        40
           1       0.71      0.39      0.50        31
           2       0.78      0.63      0.70        46
           3       0.39      0.58      0.46        33

    accuracy                           0.60       150
   macro avg       0.63      0.59      0.59       150
weighted avg       0.64      0.60      0.60       150


Tree Structure:
QUEST Decision Tree Structure:
  Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Leaves: 15
  Depth: 5
[Node] Feature 1 (continuous)
       Super-classes: [np.int64(2)] vs [np.int64(0), np.int64(1), np.int64(3)]
       Threshold: 0.2373
       n_samples: 350
  |-- Left (<=):
  |   [Node] Feature 4 (continuous)
  |          Super-classes: [np.int64(1), np.int64(2)] vs [np.int64(0

### TEST 3: Cost-Complexity Pruning Demonstration

In [18]:
# ============================================================
# TEST 3: Cost-Complexity Pruning Demonstration
# ============================================================
print("=" * 60)
print("TEST 3: Cost-Complexity Pruning")
print("=" * 60)

# Count leaves helper
def count_leaves(node):
    if node is None:
        return 0
    if node.is_leaf:
        return 1
    return count_leaves(node.left_child) + count_leaves(node.right_child)

# Apply pruning with different alpha values
print("\n[Testing different ccp_alpha values:]")
print("-" * 50)

for alpha in [0.0, 0.01, 0.02, 0.05, 0.10]:
    tree_test = QuestTree(
        max_depth=8,
        min_samples_split=5,
        feature_types=feature_types_bin,
        ccp_alpha=alpha
    )
    tree_test.fit(X_train_bin, y_train_bin)

    # Apply cost-complexity pruning using the external function
    if alpha > 0:
        tree_test.root_, _ = cost_complexity_prune(
            tree_test.root_,
            alpha,
            len(X_train_bin)
        )

    leaves = count_leaves(tree_test.root_)
    y_pred = tree_test.predict(X_test_bin)
    acc = accuracy_score(y_test_bin, y_pred)

    train_pred = tree_test.predict(X_train_bin)
    train_acc = accuracy_score(y_train_bin, train_pred)

    print(f"  α = {alpha:.2f}: leaves = {leaves:2d}, train_acc = {train_acc:.4f}, test_acc = {acc:.4f}")

print("-" * 50)
print("\nNote: Increasing α leads to more aggressive pruning (fewer leaves)")
print("      but may improve test accuracy by reducing overfitting.")

TEST 3: Cost-Complexity Pruning

[Testing different ccp_alpha values:]
--------------------------------------------------
  α = 0.00: leaves = 11, train_acc = 0.9429, test_acc = 0.8889
  α = 0.01: leaves =  4, train_acc = 0.9143, test_acc = 0.8000
  α = 0.02: leaves =  4, train_acc = 0.9143, test_acc = 0.8000
  α = 0.05: leaves =  4, train_acc = 0.9143, test_acc = 0.8000
  α = 0.10: leaves =  2, train_acc = 0.7905, test_acc = 0.7333
--------------------------------------------------

Note: Increasing α leads to more aggressive pruning (fewer leaves)
      but may improve test accuracy by reducing overfitting.


### Test on real data: Iris data

In [19]:
# ============================================================
# Final Cell (Iris dataset): Multiclass Validation (Executed as last cell)
# ============================================================
print("=" * 60)
print("FINAL TEST (Iris dataset) — placed and executed as the last cell")
print("=" * 60)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load Iris data
X_iris, y_iris = load_iris(return_X_y=True)
feature_types_iris = ['continuous'] * X_iris.shape[1]

# Train/test split (stratified)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Train QUEST tree (no pruning)
tree_iris = QuestTree(
    max_depth=5,
    min_samples_split=10,
    feature_types=feature_types_iris,
    ccp_alpha=0.0
)
tree_iris.fit(X_train_iris, y_train_iris)

# Predict and evaluate
y_pred_iris = tree_iris.predict(X_test_iris)
print(f"\nClasses detected: {tree_iris.classes_}")
print(f"Training samples: {len(X_train_iris)}")
print(f"Test samples: {len(X_test_iris)}")
print(f"\nAccuracy (no pruning): {accuracy_score(y_test_iris, y_pred_iris):.4f}")
print("\nClassification Report (no pruning):")
print(classification_report(y_test_iris, y_pred_iris))

print("\nTree Structure (no pruning):")
tree_iris.print_tree()

# Demonstrate cost-complexity pruning on the fitted tree
alpha = 0.01
print("\nApplying cost-complexity pruning with alpha=", alpha)

import copy
pruned_root, path = cost_complexity_prune(copy.deepcopy(tree_iris.root_), alpha, len(X_train_iris))

tree_iris_pruned = copy.deepcopy(tree_iris)
tree_iris_pruned.root_ = pruned_root

y_pred_pruned = tree_iris_pruned.predict(X_test_iris)
print(f"\nAccuracy (pruned, alpha={alpha}): {accuracy_score(y_test_iris, y_pred_pruned):.4f}")
print("\nClassification Report (pruned):")
print(classification_report(y_test_iris, y_pred_pruned))

print("\nPruning path summary (first 5 entries):")
for a, leaves in path[:5]:
    print(f"  alpha={a:.6f}, leaves={leaves}")

print("\nPruned Tree Structure:")
tree_iris_pruned.print_tree()

FINAL TEST (Iris dataset) — placed and executed as the last cell

Classes detected: [0 1 2]
Training samples: 105
Test samples: 45

Accuracy (no pruning): 0.8667

Classification Report (no pruning):
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        15
           1       0.86      0.80      0.83        15
           2       0.92      0.80      0.86        15

    accuracy                           0.87        45
   macro avg       0.87      0.87      0.86        45
weighted avg       0.87      0.87      0.86        45


Tree Structure (no pruning):
QUEST Decision Tree Structure:
  Classes: [np.int64(0), np.int64(1), np.int64(2)]
  Leaves: 6
  Depth: 4
[Node] Feature 0 (continuous)
       Super-classes: [np.int64(0)] vs [np.int64(1), np.int64(2)]
       Threshold: 5.2767
       n_samples: 105
  |-- Left (<=):
  |   [LEAF] Predict: 0 | n=31 | dist=(0:28, 1:3)
  |-- Right (>):
      [Node] Feature 2 (continuous)
             Super-cla

### Test with dataset of UCI Car Evaluation (categorical + ordinal, multiclass)

In [20]:
# ============================================================
# REAL DATA TEST: UCI Car Evaluation (categorical + ordinal, multiclass)
# ============================================================
print("=" * 60)
print("REAL DATA TEST: UCI Car Evaluation")
print("=" * 60)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import sys, os
import numpy as np

# Ensure project root is importable (not used for preprocessing now)
proj_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']

try:
    df = pd.read_csv(url, header=None, names=columns)
    print(f"Loaded Car dataset: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print("Failed to download Car dataset:", e)
    print("Falling back to synthetic mixed dataset (already run earlier).")
    raise

# Show value counts (brief)
print('\nSample of dataset and class distribution:')
print(df['class'].value_counts())

# Define feature types: all attributes are categorical/ordinal
feature_cols = columns[:-1]
X = df[feature_cols].values
y = df['class'].values

feature_types_car = ['categorical'] * len(feature_cols)

# Split
X_train_car, X_test_car, y_train_car, y_test_car = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- Inline preprocessing (use only code within this notebook) ---
# Determine categories for each categorical feature from training data
categorical_categories = {}
for i in range(X_train_car.shape[1]):
    cats = list(pd.Series(X_train_car[:, i]).unique())
    # keep original order but sort for deterministic behavior
    try:
        cats_sorted = sorted(cats)
    except Exception:
        cats_sorted = cats
    categorical_categories[i] = cats_sorted

# Build label encoder/decoder from training labels
unique_classes = sorted(list(set(y_train_car)))
encoder = {cls: idx for idx, cls in enumerate(unique_classes)}
decoder = {idx: cls for idx, cls in enumerate(unique_classes)}

# Transform X to object-dtype 2D arrays (keep categorical values as original)
X_train_t = np.empty(X_train_car.shape, dtype=object)
X_test_t = np.empty(X_test_car.shape, dtype=object)
for i in range(X_train_car.shape[1]):
    if feature_types_car[i] == 'continuous':
        X_train_t[:, i] = X_train_car[:, i].astype(float)
        X_test_t[:, i] = X_test_car[:, i].astype(float)
    else:
        X_train_t[:, i] = X_train_car[:, i]
        X_test_t[:, i] = X_test_car[:, i]

# Encode labels to integers using encoder
y_train_enc = np.array([encoder[val] for val in y_train_car])
y_test_enc = np.array([encoder[val] for val in y_test_car])

# --- End inline preprocessing ---

# Train QuestTree
tree_car = QuestTree(
    max_depth=8,
    min_samples_split=10,
    feature_types=['categorical'] * len(feature_cols),
    ccp_alpha=0.0
)

# Fit with transformed data (note: X arrays contain object dtype values for categories)
tree_car.fit(X_train_t, y_train_enc)

# Predict and evaluate
y_pred_car_enc = tree_car.predict(X_test_t)

# Convert encoded predictions back to original labels
y_pred_car = np.array([decoder.get(int(code), None) for code in y_pred_car_enc])

print(f"\nClasses detected (encoded): {tree_car.classes_}")
print(f"Training samples: {len(X_train_car)}")
print(f"Test samples: {len(X_test_car)}")

acc_car = accuracy_score(y_test_car, y_pred_car)
print(f"\nAccuracy (Car dataset): {acc_car:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_car, y_pred_car))

print('\nTree Structure:')
tree_car.print_tree()

# Try a small pruning alpha
alpha = 0.01
print('\nApplying cost-complexity pruning with alpha=', alpha)
import copy
pruned_root_car, path_car = cost_complexity_prune(copy.deepcopy(tree_car.root_), alpha, len(X_train_car))

tree_car_pruned = copy.deepcopy(tree_car)
tree_car_pruned.root_ = pruned_root_car

# Predict post-pruning
y_pred_car_pr = tree_car_pruned.predict(X_test_t)
y_pred_car_pr = np.array([decoder.get(int(code), None) for code in y_pred_car_pr])

print(f"\nAccuracy (pruned, alpha={alpha}): {accuracy_score(y_test_car, y_pred_car_pr):.4f}")
print('\nPruned Tree Structure:')
tree_car_pruned.print_tree()

print('\nPruning path (first 5):')
for a, leaves in path_car[:5]:
    print(f"  alpha={a:.6f}, leaves={leaves}")


REAL DATA TEST: UCI Car Evaluation
Loaded Car dataset: 1728 rows, 7 columns

Sample of dataset and class distribution:
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

Classes detected (encoded): [0 1 2 3]
Training samples: 1209
Test samples: 519

Accuracy (Car dataset): 0.9249

Classification Report:
              precision    recall  f1-score   support

         acc       0.80      0.91      0.85       115
        good       0.93      0.62      0.74        21
       unacc       0.98      0.96      0.97       363
       vgood       0.83      0.75      0.79        20

    accuracy                           0.92       519
   macro avg       0.88      0.81      0.84       519
weighted avg       0.93      0.92      0.92       519


Tree Structure:
QUEST Decision Tree Structure:
  Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Leaves: 42
  Depth: 8
[Node] Feature 3 (categorical)
       Super-classes: [np.int64(2)] vs [np.int64(0), np

# Comparison: QUEST (Python) vs rpart (R) on Iris

This cell fits the QUEST implementation from this notebook and an `rpart` model in R (via `rpy2`) on the Iris dataset, then computes a broad set of classification metrics for both models. If `rpart` or R is not available, the R fit will attempt to install `rpart` and will fallback gracefully if that fails.

In [21]:
# ============================================================
# Comparison: QUEST (Python) vs rpart (R) on Iris
# ============================================================

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score)
from sklearn.preprocessing import label_binarize

# Load Iris
iris = load_iris()
feature_cols = [f.replace(' ', '_').replace('/', '_') for f in iris.feature_names]
X = pd.DataFrame(iris.data, columns=feature_cols)
# Use human-readable class names for R and consistency
y = pd.Series(iris.target).map({i: f"class_{i}" for i in range(len(iris.target_names))})
data = pd.concat([X, y.rename('target_name')], axis=1)

# Train/test split (stratified)
train_df, test_df = train_test_split(data, test_size=0.3, random_state=42, stratify=y)

# Prepare arrays for Python QUEST (use original numeric labels for our implementation)
y_train_numeric = train_df['target_name'].map(lambda s: int(s.split('_')[-1])).values
y_test_numeric = test_df['target_name'].map(lambda s: int(s.split('_')[-1])).values
X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values

# --- Fit QUEST (this notebook implementation) ---
print('Fitting QUEST (Python implementation)...')
feature_types_iris_cmp = ['continuous'] * X_train.shape[1]
quest = QuestTree(max_depth=5, min_samples_split=10, feature_types=feature_types_iris_cmp, ccp_alpha=0.0)
quest.fit(X_train, y_train_numeric)
y_pred_quest = quest.predict(X_test)

# Compute Python QUEST metrics
def compute_metrics(y_true, y_pred, name):
    out = {}
    out['accuracy'] = accuracy_score(y_true, y_pred)
    out['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    out['precision_macro'], out['recall_macro'], out['f1_macro'], _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    out['precision_weighted'], out['recall_weighted'], out['f1_weighted'], _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    out['matthews_corrcoef'] = matthews_corrcoef(y_true, y_pred)
    out['cohen_kappa'] = cohen_kappa_score(y_true, y_pred)
    # Confusion matrix and per-class metrics
    out['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    out['per_class'] = {f'class_{i}': {
        'precision': float(prec[i]) if len(prec)>i else 0.0,
        'recall': float(rec[i]) if len(rec)>i else 0.0,
        'f1': float(f1[i]) if len(f1)>i else 0.0} for i in range(3)}
    # Multiclass ROC AUC (one-vs-rest, requires binarized labels)
    try:
        y_true_bin = label_binarize(y_true, classes=list(range(3)))
        y_pred_bin = label_binarize(y_pred, classes=list(range(3)))
        out['roc_auc_ovr_macro'] = float(roc_auc_score(y_true_bin, y_pred_bin, average='macro', multi_class='ovr'))
    except Exception:
        out['roc_auc_ovr_macro'] = None
    return out

metrics_quest = compute_metrics(y_test_numeric, y_pred_quest, 'QUEST (Python)')
print('\nQUEST (Python) metrics:')
print('  Accuracy:', metrics_quest['accuracy'])
print('  Balanced Accuracy:', metrics_quest['balanced_accuracy'])
print('  Macro F1:', metrics_quest['f1_macro'])
print('  Weighted F1:', metrics_quest['f1_weighted'])
print('  Matthews Corr:', metrics_quest['matthews_corrcoef'])
print('  Cohen Kappa:', metrics_quest['cohen_kappa'])
print('  Confusion Matrix:\n', metrics_quest['confusion_matrix'])

# --- Fit rpart in R via rpy2 ---
print('\nAttempting to fit rpart in R (via rpy2)...')
r_available = True
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
    pandas2ri.activate()
    base = importr('base')
    utils = importr('utils')
    try:
        rpart = importr('rpart')
    except Exception:
        print('rpart not found in R; attempting to install...')
        utils.install_packages('rpart')
        rpart = importr('rpart')
except Exception as e:
    print('rpy2 or R not available in this environment:', e)
    r_available = False

y_pred_r = None
metrics_r = None
if r_available:
    try:
        # Prepare DataFrames for R (features + target_name as factor)
        ro.globalenv['train_df'] = pandas2ri.py2rpy(train_df)
        ro.globalenv['test_df'] = pandas2ri.py2rpy(test_df)
        # Try to fit rpart using QUEST split option; if it fails, fallback to default rpart
        r_code = '''
        library(rpart)
        train_df$target_name <- as.factor(train_df$target_name)
        test_df$target_name <- as.factor(test_df$target_name)
        # Try fitting with QUEST-like split (some rpart builds support 'QUEST');
        quest_model <- try(rpart(target_name ~ ., data = train_df, method = 'class',
                             parms = list(split = 'QUEST'),
                             control = rpart.control(minsplit = 10, minbucket = 5, cp = 0)), silent = TRUE)
        if(inherits(quest_model, 'try-error')){
            # Fallback: fit standard rpart with default split (gini/information)
            quest_model <- rpart(target_name ~ ., data = train_df, method = 'class',
                              control = rpart.control(minsplit = 10, minbucket = 5, cp = 0))
        }
        pred <- predict(quest_model, newdata = test_df, type = 'class')
        '''
        ro.r(r_code)
        # Extract predictions
        y_pred_r = np.array(ro.r('as.character(pred)'))
        # Map R prediction strings to numeric labels consistent with our numeric mapping
        y_pred_r_numeric = np.array([int(s.split('_')[-1]) for s in y_pred_r])
        metrics_r = compute_metrics(y_test_numeric, y_pred_r_numeric, 'rpart (R)')
        print('\nR rpart metrics:')
        print('  Accuracy:', metrics_r['accuracy'])
        print('  Balanced Accuracy:', metrics_r['balanced_accuracy'])
        print('  Macro F1:', metrics_r['f1_macro'])
        print('  Weighted F1:', metrics_r['f1_weighted'])
        print('  Matthews Corr:', metrics_r['matthews_corrcoef'])
        print('  Cohen Kappa:', metrics_r['cohen_kappa'])
        print('  Confusion Matrix:\n', metrics_r['confusion_matrix'])
        # Print R tree summary
        print('\nR tree summary:')
        print(ro.r('print(quest_model)'))
    except Exception as e:
        print('Failed to run rpart in R:', e)
        r_available = False

# --- Side-by-side summary if both ran ---
print('\n\n----- Summary (side-by-side) -----')
print('QUEST (Python):')
print(f"  Accuracy: {metrics_quest['accuracy']:.4f}")
print(f"  Macro F1: {metrics_quest['f1_macro']:.4f}")
print(f"  Weighted F1: {metrics_quest['f1_weighted']:.4f}")
print(f"  Matthews Corr: {metrics_quest['matthews_corrcoef']:.4f}")
print(f"  Cohen Kappa: {metrics_quest['cohen_kappa']:.4f}")
print('')
if metrics_r is not None:
    print('rpart (R):')
    print(f"  Accuracy: {metrics_r['accuracy']:.4f}")
    print(f"  Macro F1: {metrics_r['f1_macro']:.4f}")
    print(f"  Weighted F1: {metrics_r['f1_weighted']:.4f}")
    print(f"  Matthews Corr: {metrics_r['matthews_corrcoef']:.4f}")
    print(f"  Cohen Kappa: {metrics_r['cohen_kappa']:.4f}")
else:
    print('rpart (R) was not available or failed to run in this environment.')

print('\nDetailed reports are printed above.')


Fitting QUEST (Python implementation)...

QUEST (Python) metrics:
  Accuracy: 0.8666666666666667
  Balanced Accuracy: 0.8666666666666667
  Macro F1: 0.864606657710106
  Weighted F1: 0.8646066577101059
  Matthews Corr: 0.8041806928957864
  Cohen Kappa: 0.8
  Confusion Matrix:
 [[15  0  0]
 [ 2 12  1]
 [ 1  2 12]]

Attempting to fit rpart in R (via rpy2)...

R rpart metrics:
  Accuracy: 0.8888888888888888
  Balanced Accuracy: 0.888888888888889
  Macro F1: 0.8887652947719689
  Weighted F1: 0.888765294771969
  Matthews Corr: 0.8339513040028604
  Cohen Kappa: 0.8333333333333334
  Confusion Matrix:
 [[15  0  0]
 [ 0 12  3]
 [ 0  2 13]]

R tree summary:
n= 105 

node), split, n, loss, yval, (yprob)
      * denotes terminal node

1) root 105 70 class_0 (0.33333333 0.33333333 0.33333333)  
  2) petal_length_(cm)< 2.45 35  0 class_0 (1.00000000 0.00000000 0.00000000) *
  3) petal_length_(cm)>=2.45 70 35 class_1 (0.00000000 0.50000000 0.50000000)  
    6) petal_width_(cm)< 1.55 34  1 class_1 (0.0

---
## 5. Understanding QUEST: Theory and Comparisons

### Why is QUEST Unbiased?

The key innovation of QUEST is the **separation of variable selection from split point selection**.

#### The Bias Problem in CART and C4.5

Consider how CART selects a splitting variable:
1. For each variable, try ALL possible split points
2. Select the (variable, split) pair with best impurity reduction

This creates **selection bias** because:
- A continuous variable with many unique values has more potential splits
- Even if a variable is pure noise, some split will appear good by chance
- Variables with more categories have more ways to partition

**Example**: A continuous variable with 100 unique values has 99 potential splits. By chance alone, some split will look good, giving the variable an unfair advantage over a binary categorical variable with only 1 split.

#### How QUEST Solves This

QUEST uses a **two-phase approach**:

1. **Variable Selection**: Use hypothesis tests that have consistent power across variable types
   - ANOVA for continuous (tests mean differences)
   - Chi-square for categorical (tests independence)
   - These tests don't depend on the number of possible splits

2. **Split Point Selection**: Given the selected variable, use QDA
   - This is a parametric method that doesn't search through splits
   - The split point is computed analytically

### Comparison with Other Algorithms

| Aspect | CART | C4.5 | QUEST |
|--------|------|------|-------|
| Variable Selection | Exhaustive search | Exhaustive search | Statistical tests |
| Split Selection | Exhaustive search | Exhaustive search | QDA (analytical) |
| Bias | Toward many-valued variables | Toward many-category variables | Unbiased |
| Speed | O(n * p * m) | O(n * p * m) | O(n * p) |
| Multiway Splits | No (binary only) | Yes | No (binary only) |
| Missing Values | Surrogate splits | Special category | Not directly handled |

Where: n = samples, p = features, m = average unique values per feature

### Strengths of QUEST

1. **Unbiased**: No selection bias toward variables with many possible splits
2. **Efficient**: O(n * p) per node instead of O(n * p * m)
3. **Interpretable**: Uses well-understood statistical tests
4. **Principled**: Based on statistical theory (LDA, ANOVA)

### Limitations of QUEST

1. **Binary Classification Only**: Designed for two-class problems
   - Extensions exist (e.g., CRUISE, GUIDE) for multi-class

2. **Normality Assumption**: QDA assumes normal distributions
   - Works well in practice, but may be suboptimal for highly skewed data

3. **Binary Splits Only**: Cannot produce multi-way splits
   - Unlike C4.5 which can split on all values of a categorical

4. **Missing Values**: Not directly handled in the basic algorithm
   - Extensions exist for imputation and surrogate splits

### When to Use QUEST?

QUEST is particularly suitable when:
- You have a mix of categorical and continuous features
- Features have varying numbers of unique values
- You want to avoid spurious variable selection
- Interpretability of variable importance is critical
- You have a binary classification problem


---
## 6. Summary

This notebook implemented the QUEST decision tree algorithm from scratch, following the original paper by Loh and Shih (1997).

### Key Components Implemented

1. **Algorithm 3 (Variable Selection)**
   - ANOVA F-test for continuous variables
   - Chi-square test for categorical variables
   - Levene's test as a backup
   - Bonferroni correction for multiple testing

2. **Algorithm 2 (CRIMCOORDS)**
   - One-hot encoding
   - SVD for dimensionality reduction
   - LDA for optimal category scoring

3. **Algorithm 1 (QDA Split)**
   - Quadratic discriminant analysis
   - Analytical solution for split point
   - Handling of edge cases

### References

- Loh, W.-Y., & Shih, Y.-S. (1997). Split selection methods for classification trees. *Statistica Sinica*, 7, 815-840.
- Loh, W.-Y. (2014). Fifty years of classification and regression trees. *International Statistical Review*, 82(3), 329-348.
